# r_tower — Two-tower retrieval with logQ correction

**Papers:**
- Covington, Adams, Sargin (2016) — *Deep Neural Networks for YouTube Recommendations*. Candidate generation as extreme multiclass classification: the user tower is built from the **mean of the embeddings of watched items** (the "watch vector"), the score is a dot product, training uses sampled softmax.
- Yi et al. (2019, RecSys) — *Sampling-Bias-Corrected Neural Modeling for Large Corpus Item Recommendations*. **The contribution we implement:** in-batch negatives make popular items a negative too often and bias the softmax. The fix is **logQ correction**: subtract $\log Q(\text{item})$ (the probability that the item lands in a batch, estimated by a streaming frequency estimator) from every in-batch logit BEFORE the softmax.

**What we build and train:** a two-tower retrieval model on MovieLens-100k (implicit feedback, leave-one-out by time), trained with in-batch sampled softmax and evaluated with Recall@K (== HitRate@K, since each user has a single held-out target). Three systems are compared: a popularity baseline, two-tower without correction (ablation), two-tower with logQ correction.

**Compute:** CPU, 3 epochs, a few minutes. The dataset (~5 MB) is downloaded in the setup cell.

**Metric, taken from the candidate generation literature:** Recall@10 and Recall@50 on the held-out last interaction (Covington measures exactly this: did the next watch land in the top-N).

**Parts:**
1. Watch vector: the user tower as the mean of the history (Covington).
2. In-batch sampled softmax + logQ correction (the contribution of Yi 2019).
3. Streaming frequency estimator for $Q$ (Algorithm 1 of Yi 2019).
4. Training loop: you write it and train both models.
5. Retrieval + Recall@K: apply the method and close the loop on the metric.
6. Ablation: correction vs no correction vs popularity, to see the effect directly.

## Experiments first (fill in BEFORE reading the method)

Open Yi et al. (2019), the experiments section, and answer in your own words. This is the anchor for the rest of the notebook.

> **(a) What did the results actually show, compared with what the abstract promises?** (on which corpus, how much logQ correction moved the metric, where the effect was largest)
>
> …

> **(b) The weakest experiment / how would you falsify the contribution?** (what if negatives come from a uniform sampler instead of the batch: is the correction needed at all then?)
>
> …

> **(c) Where does it break / what was NOT tested?** (how the streaming estimator behaves on cold items; sensitivity to $\alpha$)
>
> …

## Why this matters

The retrieval stage of a recommender has to pick a few hundred candidates out of millions of items within milliseconds. A full softmax over the whole catalog at every training step is out of reach, so the softmax is computed only over items **inside the batch**: each user's positive serves as a negative for every other user in the batch. This is cheap and parallel, and it has a catch.

In-batch negatives are sampled in proportion to frequency: a blockbuster lands in the batch (and therefore among the negatives) orders of magnitude more often than a niche film. In this form the sampled softmax estimates a different quantity from the full softmax we want. It systematically **penalises popular items**, because they show up as the "wrong answer" too often. The model underrates the popular items and overrates the tail.

**The idea of Yi 2019 is logQ correction.** Subtract $\log Q(\text{item})$, the log-probability that the item made it into the batch at all, from the item's logit, and the biased in-batch estimate becomes an unbiased estimate of the full softmax. This is the same correction as in noise-contrastive / sampled-softmax theory (Bengio & Senécal), except that here $Q$ has no analytic form (batches come from the live traffic distribution), so it is **estimated on the fly** with a streaming algorithm. The advantage over the alternative (sampling negatives uniformly from the whole catalog): in-batch negatives are free and already on the GPU, and the correction makes them unbiased again. You get both speed and a correct gradient.

Why MovieLens-100k: a compact public implicit-feedback benchmark on which two-tower models are routinely sanity-checked. Leave-one-out by timestamp is the standard top-K retrieval protocol for sequential recommenders: the user's last interaction is hidden as the target, everything before it is the history.

## Theory (Feynman-style)

**Task.** A user $u$ with watch history $H_u$; we want to score all items $y$ and return the top-K. Two-tower: a user tower $\phi(u)\in\mathbb{R}^d$, an item tower $\psi(y)\in\mathbb{R}^d$, score $s(u,y)=\langle\phi(u),\psi(y)\rangle$. Both towers are L2-normalised (cosine) and the logits are divided by a temperature $\tau$.

**User tower (Covington).** There is no user id embedding. The user representation is a function of the history: average the embeddings of the watched items (the "watch vector") and pass the result through an MLP:
$$\phi(u)=\mathrm{MLP}\!\Big(\tfrac{1}{|H_u|}\sum_{y\in H_u}\psi(y)\Big).$$
This lets the model generalise to new users and new histories without retraining.

**Full softmax (what we want).** The probability of the next item $y$ for user $u$:
$$P(y\mid u)=\frac{\exp s(u,y)}{\sum_{y'\in\mathcal{Y}}\exp s(u,y')}.$$
The denominator over the whole catalog $\mathcal{Y}$ is too expensive to compute.

**In-batch sampled softmax.** A batch holds $B$ items; for user $i$ the denominator runs only over the items in the batch:
$$P_B(j\mid i)=\frac{\exp s_{ij}}{\sum_{k\in B}\exp s_{ik}},\qquad s_{ij}=\langle\phi(u_i),\psi(y_j)\rangle.$$
The correct class for row $i$ is the diagonal $j=i$. Items enter $B$ in proportion to $Q(y)$, though, so $P_B$ is a **biased** estimate of $P$.

**logQ correction (the contribution of Yi 2019).** The corrected logit:
$$s^{c}_{ij}=s_{ij}-\log Q(y_j).$$
Plugging it into the softmax over $B$ gives an unbiased (Monte-Carlo) estimate of the full softmax: subtracting $\log Q$ exactly compensates for the popular $y_j$ appearing in the denominator more often. The loss is cross-entropy with diagonal labels.

**Estimating $Q$ (Algorithm 1).** $Q(y)$ is not given; it is estimated from the average number of steps between two consecutive appearances of $y$ in the stream of batches. For item $y$: $A[y]$ is the last step at which $y$ was seen; $B[y]$ is a running average of the interval between appearances. At step $t$, on seeing $y$:
$$B[y]\leftarrow(1-\alpha)\,B[y]+\alpha\,(t-A[y]),\qquad A[y]\leftarrow t,\qquad \hat Q(y)=1/B[y].$$
Seen often → short interval → large $\hat Q$ → large $\log Q$ → a stronger penalty on the logit. The loop closes.

**Metric.** Recall@K with one held-out target is HitRate@K: 1 if the hidden item is in the top-K, 0 otherwise. Items already in the history are masked out at retrieval time.

*Where my understanding has a gap:* …

❓ **Inline Question 1.** Why **subtract** $\log Q$? Why not divide the logit by $Q$ or multiply the score by something? Hint: what does the correction do to the log of the softmax weight $\exp(s_{ij})$ in the denominator?

**Your answer:** …

In [ ]:
# --- setup (ready, just run it) ---
# pip install torch numpy
import os, io, zipfile, urllib.request
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cpu"

DATA_DIR = os.path.join(os.getcwd(), "ml-100k")
URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

def load_ratings():
    path = os.path.join(DATA_DIR, "u.data")
    if not os.path.exists(path):
        print("downloading MovieLens-100k ...")
        raw = urllib.request.urlopen(URL, timeout=60).read()
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            z.extractall(os.getcwd())
    rows = []
    with open(path) as f:
        for line in f:
            u, i, r, t = line.split("\t")
            rows.append((int(u), int(i), int(r), int(t)))
    return rows  # (user, item, rating, ts)

def build_dataset():
    rows = load_ratings()
    rows = [(u, i, t) for (u, i, r, t) in rows if r >= 4]   # implicit: rating>=4 = "watch"
    users = sorted({u for u, _, _ in rows}); items = sorted({i for _, i, _ in rows})
    uid = {u: k for k, u in enumerate(users)}; iid = {i: k for k, i in enumerate(items)}
    n_items = len(items)
    by_user = defaultdict(list)
    for u, i, t in rows:
        by_user[uid[u]].append((t, iid[i]))
    train_pairs = []; test = {}; item_pop = np.zeros(n_items)
    for u, seq in by_user.items():
        seq.sort()
        items_seq = [i for _, i in seq]
        if len(items_seq) < 5:
            continue
        test_hist, test_target = items_seq[:-1], items_seq[-1]   # leave-one-out by time
        test[u] = (test_hist, test_target)
        for k in range(1, len(test_hist)):                       # predict each item from its prefix
            train_pairs.append((u, items_seq[:k], items_seq[k]))
            item_pop[items_seq[k]] += 1.0
    return train_pairs, test, n_items, item_pop

def collate(batch):
    hists = [h for (_, h, _) in batch]
    targets = torch.tensor([t for (_, _, t) in batch], dtype=torch.long)
    L = max(len(h) for h in hists)
    padded = torch.zeros(len(batch), L, dtype=torch.long)
    lens = torch.zeros(len(batch), dtype=torch.long)
    for r, h in enumerate(hists):
        padded[r, : len(h)] = torch.tensor(h, dtype=torch.long)
        lens[r] = len(h)
    return padded, lens, targets

train_pairs, test, n_items, item_pop = build_dataset()
print(f"n_items={n_items}  n_train_pairs={len(train_pairs)}  n_test_users={len(test)}")
# Expected: n_items=1447  n_train_pairs=53485  n_test_users=938


## Part 1 — Watch vector (user tower, Covington)

The user tower has no id embedding of its own: the user representation is assembled from the history. The step you implement turns a batch of variable-length (padded) histories into one representation per user by aggregating the embeddings of the items that were actually watched. The rest of the tower (MLP, item tower) is given.

In [ ]:
class TwoTower(nn.Module):
    def __init__(self, n_items, dim=64, hidden=128):
        super().__init__()
        self.dim = dim
        self.item_emb = nn.Embedding(n_items, dim)          # shared item lookup
        nn.init.normal_(self.item_emb.weight, std=0.05)
        self.user_mlp = nn.Sequential(                       # user tower on top of the watch vector
            nn.Linear(dim, hidden), nn.ReLU(),
            nn.Linear(hidden, dim),
        )

    def user_tower(self, hist_emb_mean):
        return self.user_mlp(hist_emb_mean)                 # (B, d)

    def item_tower(self, item_ids):
        return self.item_emb(item_ids)                      # (B, d)

    def watch_vector(self, hist_padded, hist_len):
        """Covington watch vector: one history representation per user.

        Args:
          hist_padded: (B, L) long — item ids of the history, right-padded
                       (with zeros) to a common length L; the real length of row r is hist_len[r].
          hist_len:    (B,) long — number of real (non-padding) items in each row.
        Returns:
          (B, d) float — the history representation of each user in item-embedding space.
        Note: padding does not affect the result (history [a,b] and the same history padded to length 5 give the same output).
        """
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
# Contract: padding does not affect the watch vector; shape (B, d).
torch.manual_seed(0)
_m = TwoTower(n_items)
# row 0: history [3,7,2]; row 1: history [5], padded to L=3
padded = torch.tensor([[3,7,2],[5,0,0]], dtype=torch.long)
lens   = torch.tensor([3,1], dtype=torch.long)
wv = _m.watch_vector(padded, lens)
assert wv.shape == (2, _m.dim), f"watch vector must have shape (B, d), got {tuple(wv.shape)}"

# padding invariance: the same row 1, padded to L=5 with more filler
padded2 = torch.tensor([[5,0,0,0,0]], dtype=torch.long)
lens2   = torch.tensor([1], dtype=torch.long)
wv1_short = wv[1]
wv1_long  = _m.watch_vector(padded2, lens2)[0]
assert torch.allclose(wv1_short, wv1_long, atol=1e-6), \
    "padding leaked into the result: use only the first hist_len positions of each row"

# ground truth for row 0: the plain mean of its three embeddings
ref0 = _m.item_emb(torch.tensor([3,7,2])).mean(0)
assert torch.allclose(wv[0], ref0, atol=1e-6), "watch vector of row 0 != the mean of its embeddings"
print("ok — watch vector is correct and invariant to padding")


❓ **Inline Question 2.** Covington averages the history embeddings. The alternatives would be the sum or the last item. What would break with the sum (think about users with very different history lengths and about the L2 normalisation of the tower further down)?

**Your answer:** …

## Part 2 — In-batch sampled softmax + logQ correction (the contribution of Yi 2019)

The core of the paper. You have a batch of $B$ (user, positive item) pairs. The score matrix is $B\times B$: the score of each user against each item. The diagonal holds the positives, everything else is in-batch negatives. Without correction this softmax estimate is biased towards penalising popular items; the correction subtracts from the logits the log-probability that the item landed in the batch, which makes the estimate unbiased.

Implement both branches (with and without correction) in one function behind a flag; later we combine them into a single ablation.

In [ ]:
def in_batch_softmax_loss(user_emb, item_emb, log_q, use_correction):
    """In-batch sampled softmax loss for a two-tower model (Yi 2019 contract).

    Args:
      user_emb: (B, d) — user tower outputs for the B users in the batch.
      item_emb: (B, d) — item tower outputs for their B positive items.
                Item j is the positive of user j and an in-batch negative for every other user.
      log_q:    (B,) — log Q(item_j): log-probability that item j landed in the batch.
      use_correction: bool — whether to apply logQ correction.
    Returns:
      scalar tensor — batch-mean cross-entropy of the classification
      "which of the B items in the batch is this user's positive", where the correct class
      of user i is item i (the diagonal); a differentiable scalar (you call .backward() on it).
    """
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# 1) shape/finiteness/grad
torch.manual_seed(1)
B, d = 6, 8
ue = torch.randn(B, d, requires_grad=True)
ie = torch.randn(B, d)
lq = torch.randn(B)
L = in_batch_softmax_loss(ue, ie, lq, use_correction=True)
assert L.dim() == 0 and torch.isfinite(L), "the loss must be a finite scalar"
L.backward()
assert ue.grad is not None and ue.grad.abs().max() > 0, "no gradient flows into user_emb — is this really a CE over the batch?"

# 2) the correction actually moves the loss (the paper's contribution is active)
torch.manual_seed(2)
ue2 = torch.randn(B, d); ie2 = torch.randn(B, d)
lq2 = torch.tensor([3.0, -3.0, 3.0, -3.0, 3.0, -3.0])   # sharply non-uniform Q
L_on  = in_batch_softmax_loss(ue2, ie2, lq2, use_correction=True)
L_off = in_batch_softmax_loss(ue2, ie2, lq2, use_correction=False)
assert not torch.allclose(L_on, L_off), "the correction does not affect the loss — are you subtracting log_q from the logits of the right item?"

# 3) identity: with a uniform log_q (all equal) the correction is a constant shift of the logits
#    along each row -> softmax is invariant -> the losses must match
lq_const = torch.full((B,), 1.234)
La = in_batch_softmax_loss(ue2, ie2, lq_const, use_correction=True)
Lb = in_batch_softmax_loss(ue2, ie2, lq_const, use_correction=False)
assert torch.allclose(La, Lb, atol=1e-5), \
    "with a uniform log_q the correction is a constant row shift and must not change the softmax/loss"
print(f"ok — in-batch loss is correct; L_on={L_on.item():.4f} != L_off={L_off.item():.4f}")


❓ **Inline Question 3.** In the test, with a **uniform** $\log Q$ (all items equally popular), the correction changes nothing. Why is this the correct behaviour? What does it tell you about the regime in which the correction is needed at all?

**Your answer:** …

## Part 3 — Streaming estimator of $Q$ (Algorithm 1, Yi 2019)

$Q(y)$, the probability that item $y$ ends up in a random batch, is not given: batches come from the live traffic distribution. Yi estimates $Q$ on the fly from the average interval between appearances of $y$ in the stream of steps: seen often → short average interval → large $\hat Q$.

Implement the two methods of the estimator: an online update of the statistics when items are observed at step $t$, and a lookup that returns $\log\hat Q$ for the requested items. The state (`A`, `B`, `seen`, `alpha`) is already set up in `__init__`.

In [ ]:
class StreamingFreqEstimator:
    """Algorithm 1 (Yi et al. 2019): online estimate of the in-batch probability Q(item)
    from the average number of steps between two appearances of the item in the stream of batches.

    State (per item):
      A[y]    — the last step t at which item y was observed (init 0).
      B[y]    — running estimate of the mean interval between appearances of y (init 1).
      seen[y] — whether y has been observed at least once (init False).
      alpha   — rate of the exponential smoothing of the interval.
    """

    def __init__(self, n_items, alpha=0.01):
        self.A = np.zeros(n_items)
        self.B = np.ones(n_items)          # init interval=1 -> Q=1 until the first real observation
        self.alpha = alpha
        self.seen = np.zeros(n_items, dtype=bool)

    def update(self, item_ids, t):
        """Record that the items were observed at step t (updates the state in place).
        Args:    item_ids: (B,) long — observed items; t: int — step number.
        Returns: None — modifies A/B/seen in place.
        Note: the exact form of the update is Algorithm 1 (see Theory, Part 3).
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def log_q(self, item_ids):
        """log Q(item) — log of the in-batch probability of the items.
        Args:    item_ids: (M,) long — items to evaluate.
        Returns: (M,) float32 — log Q(item); finite values, Q in (0, 1].
        """
        # YOUR CODE HERE
        raise NotImplementedError


In [ ]:
# Stream: item 0 every step (frequent), item 5 once every ~10 steps (rare).
# A larger alpha so that the streaming estimate converges on a short synthetic stream.
fe = StreamingFreqEstimator(n_items, alpha=0.3)
t = 0
for _ in range(200):
    fe.update([0], t); t += 1
    if t % 10 == 0:
        fe.update([5], t); t += 1

lq = fe.log_q([0, 5])
assert lq.shape == (2,) and lq.dtype == torch.float32, f"log_q: expected float32 (2,), got {tuple(lq.shape)} {lq.dtype}"
assert torch.isfinite(lq).all(), "log_q must be finite (clip Q to (0,1], guard the divisor against zero)"
assert (lq <= 1e-4).all(), "Q is a probability in (0,1], so log_q <= 0"
# Observed: log_q(frequent)=-0.272 > log_q(rare)=-2.302
assert lq[0] > lq[1] + 0.5, f"the frequent item must get a larger log_q: {lq.tolist()}"

# update actually writes the state: A[y] == the last step
fe2 = StreamingFreqEstimator(n_items)
fe2.update([4, 9], 17)
assert fe2.A[4] == 17 and fe2.A[9] == 17 and fe2.seen[4] and fe2.seen[9], \
    "after update A[y] must equal t and seen[y] must be True"
print(f"ok — log_q(frequent)={lq[0]:.3f} > log_q(rare)={lq[1]:.3f}")


❓ **Inline Question 4.** `B` is initialised with ones (→ $Q=1$), and for an item seen for the first time the interval is not updated. What does this mean for a **cold** item that has barely been seen, during the first batches: is its logit penalised strongly or weakly? Is that good or bad for retrieving the tail?

**Your answer:** …

## Part 4 — Training loop (you write it)

Put everything together: on each batch, update the $Q$ estimator, compute watch vector → user tower → item tower, L2-normalise both towers and divide the user logits by the temperature (cosine retrieval, as in Yi), compute the in-batch loss and take an optimizer step.

The normalisation and the temperature $\tau=0.07$ are given in the contract comments (this is plumbing). The core is the order: "update Q on THIS batch, then take its log_q".

In [ ]:
def train(model, train_pairs, n_items, use_correction, epochs=3, bs=256, lr=1e-2):
    """Train a two-tower model with in-batch sampled softmax.
    Args:    model: TwoTower; train_pairs: list[(hist, pos_item)]; n_items: int;
             use_correction: bool — enable logQ; epochs/bs/lr — hyperparameters.
    Returns: TwoTower — the trained model.
    Available components: collate, freq (the Q estimator), model.watch_vector/user_tower/item_tower,
    in_batch_softmax_loss, opt (Adam); the score is cosine with temperature TEMP. Print the loss per epoch, shuffle the pairs.
    Note: the Q estimator must count the current batch BEFORE you take log_q for it (otherwise the estimate lags one step behind).
    """
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    freq = StreamingFreqEstimator(n_items)
    TEMP = 0.07
    step = 0
    order = np.arange(len(train_pairs))
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Smoke training on a subsample: the loss must go down. (The full run is in Part 5.)
torch.manual_seed(0); np.random.seed(0)
sub = train_pairs[:6000]
m_smoke = TwoTower(n_items)
import contextlib, io as _io
buf = _io.StringIO()
with contextlib.redirect_stdout(buf):
    train(m_smoke, sub, n_items, use_correction=True, epochs=3)
import re
losses = [float(x) for x in re.findall(r"loss=([0-9.]+)", buf.getvalue())]
assert len(losses) >= 3, "train must print the mean loss for each epoch (3 numbers)"
assert losses[-1] < losses[0], f"loss does not go down over 3 epochs: {losses} — order of the Q update / normalisation / temperature?"
# a gradient actually flowed at least once
assert any(p.grad is not None for p in m_smoke.parameters()), "no parameter has a gradient — was backward called?"
print(f"ok — loss goes down {losses[0]:.3f} -> {losses[-1]:.3f}")


## Part 5 — Retrieval + Recall@K (applying the method)

Now the inference itself: for each test user, build the user vector from the history, score ALL items in the catalog, **mask the items already seen**, take the top-K and check whether the hidden target is among them. This is candidate generation at work: a real retrieval over the whole catalog.

`all_item_vectors` (the normalised matrix of all item vectors) is given.

In [ ]:
@torch.no_grad()
def all_item_vectors(model):
    ids = torch.arange(model.item_emb.num_embeddings)
    return F.normalize(model.item_tower(ids), dim=-1)        # (N, d)

@torch.no_grad()
def recall_at_k(model, test, ks=(10, 50)):
    """Recall@K (== HitRate@K) of a trained two-tower model on the leave-one-out test.

    Args:
      test: dict user -> (hist, target), where hist is the list of history ids,
            target is the hidden last item.
      ks:   which K values to compute.
    Returns:
      dict {K: recall@K}, where recall@K is the fraction of test users (with a non-empty
      history) whose hidden target is in the model's top-K items.
      Skip users with an empty history; exclude already seen items from the ranking.
    Note: build the user representation with the same user tower/normalisation as in training (otherwise train and eval diverge).
    """
    item_vecs = all_item_vectors(model)
    # YOUR CODE HERE
    raise NotImplementedError

def recall_at_k_popularity(test, item_pop, ks=(10, 50)):
    """Baseline: recommend the most frequent items to everyone (minus the seen ones)."""
    order = np.argsort(-item_pop)
    hits = {k: 0 for k in ks}; n = 0
    for u, (hist, target) in test.items():
        if len(hist) == 0:
            continue
        seen = set(hist)
        rec = [int(i) for i in order if int(i) not in seen]
        n += 1
        for k in ks:
            if target in rec[:k]:
                hits[k] += 1
    return {k: hits[k] / max(n, 1) for k in ks}


In [ ]:
# Check recall@k on a small trained model + the masking contract.
torch.manual_seed(0); np.random.seed(0)
m_eval = TwoTower(n_items)
import contextlib, io as _io
with contextlib.redirect_stdout(_io.StringIO()):
    train(m_eval, train_pairs[:6000], n_items, use_correction=True, epochs=2)

r = recall_at_k(m_eval, test, ks=(10, 50))
assert set(r.keys()) == {10, 50}, "recall_at_k must return a dict keyed by ks"
assert 0.0 <= r[10] <= 1.0 and 0.0 <= r[50] <= 1.0, "recall is a fraction in [0,1]"
assert r[50] >= r[10], "R@50 cannot be lower than R@10 (top-50 contains top-10)"

# masking contract: seen items must not appear in the ranking.
# take a user and check that their target, if it HAPPENS to be in the history, is not "counted" via the history
item_vecs = all_item_vectors(m_eval)
u0 = next(u for u, (h, t) in test.items() if len(h) > 0)
h0, _ = test[u0]
padded = torch.tensor(h0, dtype=torch.long).unsqueeze(0)
lens = torch.tensor([len(h0)], dtype=torch.long)
uvec = F.normalize(m_eval.user_tower(m_eval.watch_vector(padded, lens)), dim=-1)
scores = (uvec @ item_vecs.t()).squeeze(0).clone()
scores[torch.tensor(h0, dtype=torch.long)] = -1e9
top = torch.topk(scores, 50).indices.tolist()
assert not (set(h0) & set(top)), "seen items leaked into the top-K — mask the history before topk"
print(f"ok — recall@k works; on the under-trained model R@10={r[10]:.4f} R@50={r[50]:.4f}")


### Full run: train both models and the popularity baseline

A ready-made cell: we train two-tower **with** and **without** correction (the ablation) and compute three sets of Recall@K. A few minutes on CPU.

In [ ]:
torch.manual_seed(0); np.random.seed(0)

print("training two-tower WITH logQ correction (Yi 2019) ...")
model_c = TwoTower(n_items).to(DEVICE)
train(model_c, train_pairs, n_items, use_correction=True, epochs=3)

print("training two-tower WITHOUT correction (ablation) ...")
model_u = TwoTower(n_items).to(DEVICE)
train(model_u, train_pairs, n_items, use_correction=False, epochs=3)

rec_c   = recall_at_k(model_c, test)
rec_u   = recall_at_k(model_u, test)
rec_pop = recall_at_k_popularity(test, item_pop)

print("\n--- Recall@K (== HitRate@K) ---")
print(f"popularity baseline : R@10={rec_pop[10]:.4f}  R@50={rec_pop[50]:.4f}")
print(f"two-tower (no corr) : R@10={rec_u[10]:.4f}  R@50={rec_u[50]:.4f}")
print(f"two-tower (+logQ)   : R@10={rec_c[10]:.4f}  R@50={rec_c[50]:.4f}")
# Expected (seed 0): pop 0.0501/0.1684 | no-corr 0.0416/0.2004 | +logQ 0.0832/0.2772


In [ ]:
# Result contract: the method beats the baseline, and the correction helps.
# Thresholds are calibrated with a margin on real runs (seed 0/7).
print(f"+logQ   R@10={rec_c[10]:.4f}  R@50={rec_c[50]:.4f}")
print(f"no-corr R@10={rec_u[10]:.4f}  R@50={rec_u[50]:.4f}")
print(f"pop     R@10={rec_pop[10]:.4f}  R@50={rec_pop[50]:.4f}")

assert rec_c[50] >= 0.25, f"R@50 of the method {rec_c[50]:.4f} is below the expected ~0.277 — under-training/retrieval bug"
assert rec_c[10] >= 0.07, f"R@10 of the method {rec_c[10]:.4f} is below the expected ~0.083"
assert rec_c[50] > rec_pop[50] + 0.08, \
    f"two-tower must clearly beat popularity on R@50 (observed gap ~0.11): {rec_c[50]:.4f} vs {rec_pop[50]:.4f}"
# ablation: logQ correction must help (the paper's contribution); observed 0.083 vs 0.042
assert rec_c[10] > rec_u[10], \
    f"logQ correction must lift R@10 above the ablation: {rec_c[10]:.4f} (corr) vs {rec_u[10]:.4f} (no-corr)"
print("ok — the method beats popularity, and logQ correction helps")


## Key figure: the three systems side by side

The headline result, computed by your functions. Compare it with what Yi 2019 reports: on their corpus logQ correction consistently improves the retrieval metric, and the ablation without correction loses ground specifically on popular/head traffic. On MovieLens we see the same sign of the effect.

In [ ]:
import matplotlib.pyplot as plt
labels = ["popularity", "two-tower\n(no corr)", "two-tower\n(+logQ)"]
r10 = [rec_pop[10], rec_u[10], rec_c[10]]
r50 = [rec_pop[50], rec_u[50], rec_c[50]]
x = np.arange(3); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - w/2, r10, w, label="Recall@10")
ax.bar(x + w/2, r50, w, label="Recall@50")
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel("Recall@K (== HitRate@K)")
ax.set_title("Candidate generation on MovieLens-100k (leave-one-out)")
ax.legend(); ax.grid(True, axis="y", alpha=0.3)
for xi, v in zip(x - w/2, r10): ax.text(xi, v + 0.003, f"{v:.3f}", ha="center", fontsize=8)
for xi, v in zip(x + w/2, r50): ax.text(xi, v + 0.003, f"{v:.3f}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()
print("in the paper logQ correction is consistently > the ablation; same sign here: +logQ beats both no-corr and popularity")


## Probing the effect: where exactly the correction wins

The ablation is already built into the run: the only difference between `model_c` and `model_u` is the `use_correction` flag. That is the natural contrast in which the advantage shows. Let us look at the effect where the theory predicts it: on **popular** items. Without correction, in-batch negatives teach the model to suppress hits; the correction gives them back a fair logit.

In [ ]:
# Take the most popular test targets and compute R@50 on them separately:
# this is where the correction should undo the in-batch negative bias.
pop_rank = {int(i): r for r, i in enumerate(np.argsort(-item_pop))}
pop_thresh = np.percentile([pop_rank[t] for (_, (_, t)) in test.items()], 25)  # top 25% by target popularity

def recall_subset(model, only_popular):
    iv = all_item_vectors(model); hit = n = 0
    for u, (hist, target) in test.items():
        if len(hist) == 0: continue
        is_pop = pop_rank[int(target)] <= pop_thresh
        if is_pop != only_popular: continue
        padded = torch.tensor(hist, dtype=torch.long).unsqueeze(0)
        lens = torch.tensor([len(hist)], dtype=torch.long)
        uvec = F.normalize(model.user_tower(model.watch_vector(padded, lens)), dim=-1)
        sc = (uvec @ iv.t()).squeeze(0).clone()
        sc[torch.tensor(hist, dtype=torch.long)] = -1e9
        n += 1; hit += int(target in torch.topk(sc, 50).indices.tolist())
    return hit / max(n, 1), n

for only_pop in (True, False):
    rc, n = recall_subset(model_c, only_pop); ru, _ = recall_subset(model_u, only_pop)
    seg = "popular targets" if only_pop else "tail targets"
    print(f"{seg:20s} (n={n:4d}): +logQ R@50={rc:.4f} | no-corr R@50={ru:.4f} | delta={rc-ru:+.4f}")


❓ **Inline Question 5 (recognition / transfer).** Picture your own problem: retrieval over a corpus with a **heavy power-law popularity tail** and in-batch negatives (products, videos, music). In which new situation would you reach for logQ correction first, and which symptom in the metrics would tell you that in-batch negative bias is already hurting quality?

**Your answer:** …

## Debug area

Fragile spots and how to catch them before full training:

- **watch_vector**: run a forward pass on a dummy input BEFORE training: `TwoTower(n_items).watch_vector(padded, lens)` must return `(B, d)` and be invariant to padding (the Part 1 test). The most common bug is dividing by `L` (the padded length) instead of `hist_len`.
- **in_batch_softmax_loss**: check it with `B=2`: `2×2` logits, labels `[0,1]`. Compute the CE by hand with unit embeddings. If the correction "does not move the loss", you are subtracting `log_q` along the wrong dimension (it has to be the item column, not the user row).
- **StreamingFreqEstimator**: the order in train is `update` at the current step, then `log_q`. If you swap them, the current batch is not counted and the estimate lags behind.
- **train**: if the loss does not go down, you forgot `opt.zero_grad()`; or you did not divide the user logits by the temperature (cosines in `[-1,1]` give an almost flat softmax and a tiny gradient); or you normalise along a dimension other than `dim=-1`.
- **recall_at_k**: if R@K looks suspiciously high, check the history masking (seen items leaking into the top-K) and that the user vector is built the same way as in training.

## Going deeper (bonus, no points)

1. **Sensitivity to $\alpha$.** Run the $Q$ estimator with `alpha` $\in\{0.001, 0.01, 0.1\}$ and look at the spread of $\log Q$ and at the final Recall. Where is the trade-off between "adapts quickly" and "noisy"?
2. **Temperature $\tau$.** Sweep `TEMP` $\in\{0.03, 0.07, 0.2\}$. How do the sharpness of the softmax and the metric change?
3. **nDCG@K instead of Recall@K.** A single held-out target makes Recall == HitRate; hold out several items (last-N) and compute nDCG. Does the sign of the correction's gain change?
4. **Uniform negatives instead of in-batch.** Replace in-batch negatives with a sample from the whole catalog. Is logQ correction still needed? (Connect this to Inline Question 3.)

> Experiment log: …

## SPOILERS — full solutions (open ONLY when you are done or stuck)

Try on your own first. The hints in the docstrings and in the Debug area are usually enough.

<details>
<summary>Part 1 — watch_vector</summary>

```python
def watch_vector(self, hist_padded, hist_len):
    emb = self.item_emb(hist_padded)                                  # (B, L, d)
    mask = (torch.arange(hist_padded.size(1))[None, :] < hist_len[:, None])
    emb = emb * mask.unsqueeze(-1)
    summed = emb.sum(1)
    return summed / hist_len.clamp(min=1).unsqueeze(-1).float()
```
</details>

<details>
<summary>Part 2 — in_batch_softmax_loss</summary>

```python
def in_batch_softmax_loss(user_emb, item_emb, log_q, use_correction):
    logits = user_emb @ item_emb.t()                 # (B, B)
    if use_correction:
        logits = logits - log_q.unsqueeze(0)         # subtract log Q along the item column
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)
```
</details>

<details>
<summary>Part 3 — StreamingFreqEstimator.update / log_q</summary>

```python
def update(self, item_ids, t):
    for y in item_ids:
        if self.seen[y]:
            self.B[y] = (1 - self.alpha) * self.B[y] + self.alpha * (t - self.A[y])
        self.A[y] = t
        self.seen[y] = True

def log_q(self, item_ids):
    q = 1.0 / np.maximum(self.B[item_ids], 1e-6)
    q = np.clip(q, 1e-6, 1.0)
    return torch.tensor(np.log(q), dtype=torch.float32)
```
</details>

<details>
<summary>Part 4 — train (body after the given setup)</summary>

```python
    for ep in range(epochs):
        np.random.shuffle(order)
        total = 0.0; nb = 0
        for s in range(0, len(order), bs):
            idx = order[s : s + bs]
            batch = [train_pairs[j] for j in idx]
            padded, lens, targets = collate(batch)
            freq.update(targets.tolist(), step); step += 1
            log_q = freq.log_q(targets.tolist())
            wv = model.watch_vector(padded, lens)
            u  = F.normalize(model.user_tower(wv), dim=-1) / TEMP
            it = F.normalize(model.item_tower(targets), dim=-1)
            loss = in_batch_softmax_loss(u, it, log_q, use_correction)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item(); nb += 1
        print(f"  epoch {ep+1}: loss={total/nb:.4f}")
    return model
```
</details>

<details>
<summary>Part 5 — recall_at_k (body after all_item_vectors)</summary>

```python
    hits = {k: 0 for k in ks}; n = 0
    for u, (hist, target) in test.items():
        if len(hist) == 0:
            continue
        padded = torch.tensor(hist, dtype=torch.long).unsqueeze(0)
        lens = torch.tensor([len(hist)], dtype=torch.long)
        wv = model.watch_vector(padded, lens)
        uvec = F.normalize(model.user_tower(wv), dim=-1)
        scores = (uvec @ item_vecs.t()).squeeze(0).clone()
        scores[torch.tensor(hist, dtype=torch.long)] = -1e9    # mask what was already seen
        n += 1
        for k in ks:
            if target in torch.topk(scores, k).indices.tolist():
                hits[k] += 1
    return {k: hits[k] / max(n, 1) for k in ks}
```
</details>


## Reflection

**What to take into your own work.** The trick "estimate the sampling bias and subtract its log from the logit" is general: whenever training negatives come from a distribution different from the target one (which is almost always the case when negatives are cheap), a cheap additive correction to the logit restores unbiasedness, with no need for expensive resampling. Streaming estimation of an unknown $Q$ from the intervals between appearances is a separate transferable trick for any online frequency counter that cannot afford a second pass.

**In which NEW situation this method is the right tool.** Any large-corpus retrieval with in-batch or popularity-proportional negatives and a heavy tail: product recommendations, search, dense retrieval for RAG (in-batch negatives there are also biased towards frequent passages). The symptom that identifies the problem: the metric on the head falls behind popularity, while on the tail it is suspiciously high, meaning the model has learned to suppress hits.

**What is still unclear → what to read next.** How logQ correction interacts with hard-negative mining: if negatives are picked by hardness, which $Q$ should be subtracted? And full sampled softmax with a learned temperature / mixed negatives is a separate topic.